# 36 (PW) — Observe & Optimize

**Production workflow, step 5.** Make the pipeline trustworthy and fast: execution plans, lineage, per-query metrics, caching, and SQL transparency. Grounded in `docs/performance_guide.md` and `docs/architecture_guide.md`.

In [ ]:
import os
from dotenv import load_dotenv
from irispark import IrisParkSession

load_dotenv()

# Connection via environment variables (matches examples/basic_usage.py).
# Set IRIS_HOST / IRIS_PORT / IRIS_NAMESPACE / IRIS_USERNAME / IRIS_PASSWORD.
try:
    session = IrisParkSession.builder() \
        .host(os.environ.get("IRIS_HOST", "localhost")) \
        .port(int(os.environ.get("IRIS_PORT", 1972))) \
        .namespace(os.environ.get("IRIS_NAMESPACE", "USER")) \
        .username(os.environ.get("IRIS_USERNAME", "_SYSTEM")) \
        .password(os.environ.get("IRIS_PASSWORD", "SYS")) \
        .getOrCreate()
    print("Connected to IRIS:", session)
except Exception as e:
    print("SKIP: IRIS not reachable -", e)
    session = None

In [ ]:
if session is None:
    raise SystemExit("IRIS not reachable; skipping this notebook.")

## 1. Working set

A small synthetic table to exercise plans and metrics.

In [ ]:
import random
import pandas as pd

random.seed(36)
vendas = session.createDataFrame(pd.DataFrame({
    "pedido_id": range(1, 501),
    "estado": [random.choice(["SP", "RJ", "MG"]) for _ in range(500)],
    "valor": [round(random.uniform(10, 500), 2) for _ in range(500)],
}))
base = vendas.filter("valor > 50")

## 2. Execution plan

`explain()` shows generated SQL + IRIS plan; `extended=True` adds logical plan and engine mapping.

In [ ]:
base.groupBy("estado").agg({"valor": "sum"}).explain()

In [ ]:
base.groupBy("estado").agg({"valor": "sum"}).explain(extended=True)

## 3. Lineage

Replay the transformation recipe for debugging.

In [ ]:
vendas.filter("valor > 100").groupBy("estado").count().lineage(show=True)

## 4. SQL transparency

`to_sql()` shows exactly what runs in IRIS.

In [ ]:
print(base.filter("estado = 'SP'").select("pedido_id", "valor").to_sql())

## 5. Per-query metrics

Opt-in observability records timing/row metrics.

In [ ]:
session.config("irispark.observability", True)
base.groupBy("estado").count().to_pandas()
metrics = getattr(session, "_metrics", [])
print("metrics recorded:", len(metrics))
for m in metrics[-2:]:
    print(m)
session.config("irispark.observability", False)

## 6. Cache hot results

`cache()` materializes once; repeated actions stop re-scanning.

In [ ]:
cached = base.cache()
print("run 1:", cached.groupBy("estado").count().orderBy("estado").collect())
print("run 2 (cached):", cached.groupBy("estado").count().orderBy("estado").collect())
cached.unpersist()
print("unpersisted")

## 7. Performance rules of thumb

From `docs/performance_guide.md`: keep filters pushable (plain column comparisons), avoid correlated scalar subqueries per group, prefer single-pass analytics (`median`, window functions) over nested loops, and use columnar storage for wide analytical scans.

In [ ]:
print("keep filters pushable; prefer single-pass analytics; use columnar for wide scans")

In [ ]:
if session is not None:
    session.close()
    print("Session closed.")